In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
import seaborn as sns

df = sns.load_dataset("penguins")
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


In [3]:
df = df.dropna()

In [4]:
y = df["species"]
X = df.drop(columns=["species"])

In [5]:
print(df["species"].unique())


['Adelie' 'Chinstrap' 'Gentoo']


# Creating Decision Tree

In [88]:

def getsplit(X, column):
    # we will make a list that contains every possible split for this column 
    values = X[column].unique()
    if values.dtype == "object":
        return values.tolist()
    else:
        values = sorted(values)
        split = []
        for i in range(len(values)-1):
            avg = (values[i]+values[i+1])/2
            split.append(round(avg, 2))
        return split


In [89]:
def split_data(df, column, weights, value):
    # if it is an object we will split the data : the left will take the lines where col == value and right will take the rest
    if df[column].dtype == "object":
        left = df[df[column] == value]
        left_weights = weights[df[column] == value]
        right = df[df[column] != value]
        right_weights = weights[df[column] != value]
    # if it is an int/float we will split the data : the left will take the lines where col < value and right will take the rest
    else:
        left = df[df[column] < value]
        left_weights = weights[df[column] < value]
        right = df[df[column] >= value]
        right_weights = weights[df[column] >= value]
    return (left, right, left_weights, right_weights)

In [90]:
def gini_impurity(weights, df, target):
    # we modify the gi so that we can callculate the weights instead of the number of samples
    class_weights = {}
    for weight, key in zip(weights, df[target]):
        if key not in class_weights:
            class_weights[key] = weight
        else :
            class_weights[key] += weight
    total = sum(weights)
    gi = 1
    for key in class_weights :
        probability = class_weights[key] / total
        gi -= probability ** 2
    return gi
    

In [91]:
def weighted_gini(left, right,left_weights, right_weights, target):
    sum_left, sum_right = sum(left_weights), sum(right_weights)
    total = sum_left + sum_right
    total_gi = (sum_left / total) * gini_impurity(left_weights, left, target) + (sum_right / total) * gini_impurity(right_weights, right, target)
    return total_gi

In [92]:
def best_split(df, target, weights):
    # we intialize the variables
    best_wg, best_split, best_column = float("inf"), None, None
    X = df.drop(target, axis = 1)
    features = list(X.columns)
    for column in features:
        # for every column we will get a list of the possible splits
        splits = getsplit(X, column)
        for split in splits:
            # for every split we will split it again so that it would be left, right respecting the rules that we mentionned before
            left, right, left_weights, right_weights = split_data(df, column, weights, split)
            if len(left) == 0 or len(right) == 0:
                continue
            wg = weighted_gini(left, right,left_weights, right_weights, target)
            # choosing the best split (the one that has the least wg)
            if wg < best_wg :
                best_wg = wg
                best_split = split
                best_column = column
    return best_wg, best_split, best_column

In [93]:
def should_stop(df, target, depth):
    if  depth >= 1 : # max depth reached
        return True
    return False

In [94]:
def create_leaf(df, target, weights):
    # we create a leaf and the biggest weight 
    class_weights = {}
    # we count weights for every key that appears
    for weight, key in zip(weights, df[target]):
        if key not in class_weights:
            class_weights[key] = weight
        else :
            class_weights[key] += weight
    max_weight = float("-inf")
    max_apperance_key = None
    # we take the one that has the biggest weight 
    for key in class_weights :
        if class_weights[key]>max_weight:
           max_weight = class_weights[key]
           max_apperance_key = key
    return {"type": "leaf" , "prediction" : max_apperance_key}
        

In [114]:
def decision_stump(df, target, weights):
    #  function that build a decision stump (tree with depth=1)
    best_wg, best_value, best_column = best_split(df, target, weights)
    if best_column is None:
        return create_leaf(df, target, weights)
    left, right, left_weights, right_weights = split_data(df, best_column, weights, best_value)
    left_tree = create_leaf(left, target, left_weights)
    right_tree = create_leaf(right, target, right_weights)
    return {
    "type": "stump",
    "column": best_column,
    "value": best_value,
    "left_prediction": left_tree,
    "right_prediction": right_tree
    }

# Creating Adaboost

In [115]:
def initialize_weights(train):
    n_samples = len(train)
    weights = np.ones(n_samples) / n_samples
    return weights

In [116]:
def weighted_error(y_true, y_pred, weights):
    # calculate the weights of the samples that are missclassified
    n = len(weights)
    error = 0
    for i in range(n):
        if y_true.iloc[i] != y_pred[i] :
            error += weights[i]
    return error

In [159]:
def compute_model_weight(error):
    # we calculate how much influence this stump should have in the final ensemble
    # we use this to protect against error = 0 or 1
    error = max(error, 1e-10)
    error = min(error, 1 - 1e-10)
    #we didn't use this alpha = (1/2) * np.log((1-error) / error) because the formula is np.log((1-error) / error)+ np.log(nb_class)
    alpha = np.log((1-error) / error)+ np.log(2)
    return alpha

In [160]:
def update_weights(weights, y_true, y_pred, alpha):
    n = len(weights)
    for i in range(n):
        if y_true.iloc[i] == y_pred[i] :
            weights[i] = weights[i] * np.exp(-alpha)
        else : 
            weights[i] = weights[i] * np.exp(alpha)
    return weights

In [161]:
def normalize_weights(weights):
    # we normalize weights(making the sum = 1)
    s = sum(weights)
    return weights / s

In [162]:
def train_adaboost(train, target, n_estimators):
    X = train.drop(target, axis=1)
    y_true = train[target]
    weights = initialize_weights(train)
    stumps = []
    alphas = []
    for i in range(n_estimators):
        # creating the model
        stump = decision_stump(train, target, weights)
        y_pred = predict_stump(stump, X)
        # calculating error
        error = weighted_error(y_true, y_pred, weights)
        alpha = compute_model_weight(error)
        weights = update_weights(weights, y_true, y_pred, alpha)
        weights = normalize_weights(weights)
        stumps.append(stump)
        alphas.append(alpha)
    return (stumps, alphas)

# Predicting result

In [163]:
def predict_stump(stump, X):
    # predict multiple lines using stump
    predictions = []
    for _, row in X.iterrows():
        predictions.append(predict_using_one_tree(stump, row))
    return predictions

In [164]:
def predict_using_one_stump(stump, sample):
    column = stump["column"]
    value = stump["value"]

    if isinstance(value, (int, float, np.number)): # we see if the value is numerical or not
        if sample[column] < value :
            return stump["left_prediction"]["prediction"] 
        else:
            return stump["right_prediction"]["prediction"]
    else:
        if sample[column] == value :
            return stump["left_prediction"]["prediction"] 
        else:
            return stump["right_prediction"]["prediction"]
        
    

In [165]:
def accuracy(y_true, y_pred):
    correct = 0
    for true, pred in zip(y_true, y_pred):
        if true == pred:
            correct += 1
        
    return correct / len(y_true)

In [166]:
def predict_one_sample(stumps, alphas, sample):
    # we used voting instead of calculating sign(sum(a * h(x))) because the target has object labels
    votes = {}
    for stump, alpha in zip(stumps, alphas):
        label = predict_using_one_stump(stump, sample)
        if label not in votes:
            votes[label] = alpha
        else:
            votes[label] += alpha
    prediction = max(votes, key=votes.get)
    return prediction

In [167]:
def predict_adaboost(stumps, alphas, X):
    predictions = []
    for _, sample in X.iterrows():
        predictions.append(predict_one_sample(stumps, alphas, sample))
    return predictions

In [168]:
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index)
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

In [169]:
stumps, alphas = train_adaboost(train, "species", 100)


In [170]:
X_test = test.drop("species", axis=1)
y_test_me = test["species"]

predictions = predict_adaboost(stumps, alphas, X_test)


In [171]:
print(accuracy(y_test_me, predictions))

0.9850746268656716


# Creating the same model using sklearn

In [172]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

X = df.drop("species", axis=1)
y = df["species"]
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
X["island"] = encoder.fit_transform(X["island"])
X["sex"] = encoder.fit_transform(X["sex"])
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
estimator = DecisionTreeClassifier(max_depth=1)

model = AdaBoostClassifier(
    estimator=estimator,
    n_estimators=100
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


# my model vs scikit-learn 

In [173]:
print(f" my accuracy :{accuracy(y_test_me, predictions)} vs scikit-learn's :{accuracy_score(y_test, y_pred)}")

 my accuracy :0.9850746268656716 vs scikit-learn's :0.9850746268656716


# Conclusion : it gives the same accuracy